# design generator

this notebook builds the genetic algorithm that generates and evolves rocket designs.

the GA sits at the top of the pipeline:

```
generate candidate        ← this notebook
    → validate_rocket     (structural — src/structure.py)
    → filter_rocket       (analytic — src/filters.py)
    → [KSP simulation]    (ground truth — later)
```

---

## what we're building

six functions, built in order:

| function | purpose |
|----------|---------|
| `generate_random_rocket` | produce a random rocket dict with no validation — GA learns from zero |
| `score_rocket` | fitness function — returns `compute_delta_v` if rocket passes all checks, else 0 |
| `tournament_select` | pick survivors from a population by tournament |
| `mutate` | apply one random valid mutation to a rocket |
| `crossover` | stage-level crossover between two parent rockets |
| `run_ga` | the main loop — initialize, evaluate, select, reproduce, repeat |

---

## key design decisions

**fitness = delta-v (or 0).** rockets that fail the structural validator or analytic filters score 0. no partial credit for near-misses. the GA starts from completely random designs and learns validity through selection pressure.

**selection = tournament.** pick M random candidates, keep the best one, repeat. preserves population diversity better than fitness-proportionate (roulette wheel) selection, and doesn't require normalized fitness scores.

**crossover = stage-level.** take upper stage(s) from parent A and lower stage(s) from parent B. invalid offspring are discarded — validation is instantaneous so this costs nothing.

---

## setup

In [ ]:
import sys
sys.path.insert(0, '...')

from src.config import load_parts_by_name, load_resource_lookup

parts_by_name = load_parts_by_name()
resource_lookup = load_resource_lookup()



In [ ]:
print(parts_by_name['mk1-3pod']['nodes'])
print(parts_by_name['fuelTank']['nodes'])
print(parts_by_name['liquidEngine']['nodes'])

generate random rocket

In [ ]:
pods = []
tanks = []
engines = []
decouplers = []
for part in parts_by_name:
    if parts_by_name[part]['category'] == 'Pods':
        pods.append(part)
    if parts_by_name[part]['resources'] is not None and parts_by_name[part]['engine'] is None:
        tanks.append(part)
    if parts_by_name[part]['engine'] is not None:
        engines.append(part)
    if part.startswith('Decoupler_'):
        decouplers.append(part)


print(len(pods))
print(len(tanks))
print(len(engines))
print(len(decouplers))

In [ ]:
import random
from src.rocket import Rocket

def generate_random_rocket(parts_by_name: dict,
                           pods: list,
                           tanks: list,
                           engines: list,
                           decouplers: list,
                           max_stages:int = 2):

    r = Rocket(parts_by_name)

    pod = random.choice(pods)
    r.add_part('pod_0', pod, parent = None)

    tank_count = engine_count = decoupler_count = 0

    n_stages = random.randint(1, max_stages)
    current_parent = 'pod_0'
    for stage in range(n_stages):
        stage_num = stage
        tank = random.choice(tanks)
        r.add_part(f'tank_{tank_count}', tank, parent = current_parent, attach_node='bottom')
        current_parent = f'tank_{tank_count}'
        tank_count += 1

        engine = random.choice(engines)
        r.add_part(f'eng_{engine_count}', engine, parent = current_parent, attach_node='bottom')
        r.set_stage(f'eng_{engine_count}', stage_num)
        current_parent = f'eng_{engine_count}'
        engine_count += 1
        if stage != n_stages - 1:
            decup = random.choice(decouplers)
            r.add_part(f'decoupler_{decoupler_count}', decup, parent = current_parent, attach_node='bottom')
            r.set_stage(f'decoupler_{decoupler_count}', stage + 1)
            current_parent = f'decoupler_{decoupler_count}'
            decoupler_count += 1

    return r.to_dict()


generate_random_rocket(parts_by_name, pods, tanks, engines, decouplers)

score rocket

In [ ]:
from src.structure import validate_rocket
from src.filters import filter_rocket, DV_THRESHOLDS, compute_delta_v

DV_THRESHOLDS = DV_THRESHOLDS


def score_rocket(rocket_dict: dict,
                 parts_by_name: dict,
                 resource_lookup: dict):

    parts_list = [p['type'] for p in rocket_dict['parts']]
    is_valid = validate_rocket(rocket_dict, parts_by_name)
    is_filtered, errors = filter_rocket(rocket_dict, parts_list, parts_by_name, resource_lookup, DV_THRESHOLDS)
    if not is_valid or not is_filtered:
        return 0
    score = compute_delta_v(rocket_dict, parts_list, parts_by_name, resource_lookup)
    return score

rocket = generate_random_rocket(parts_by_name, pods, tanks, engines, decouplers)
score = score_rocket(rocket, parts_by_name, resource_lookup)
print(rocket)
print(score)
